# 02. Building Graph Snapshots (Human, Mouse, Pig) — Your One-Time Setup

*Last updated:* 2026-01-08

This notebook shows how to **build and cache** the IDTrack graph snapshot for:
- `homo_sapiens` (human)
- `mus_musculus` (mouse)
- `sus_scrofa` (pig)

A graph build is the most expensive step. The good news:
- you usually do it **once per organism + snapshot release + YAML configuration**
- then you reuse the cached graph for fast conversions

> Prerequisite: run `prepare_new_external_yaml.ipynb` first (especially important for mouse and pig).


## 1. What you should expect (time / disk)

Graph building can take:
- minutes to hours (depends on organism, enabled externals, and cache status)
- multiple GB of disk for cached tables + the graph pickle

Plan for this like you would plan for downloading a reference genome + annotation.


In [ ]:
# 1) Setup
from __future__ import annotations

import os
from pathlib import Path

import idtrack

LOCAL_REPOSITORY = Path(os.environ.get('IDTRACK_LOCAL_REPO', './idtrack_cache')).resolve()
LOCAL_REPOSITORY.mkdir(parents=True, exist_ok=True)

api = idtrack.API(local_repository=str(LOCAL_REPOSITORY))
api.configure_logger()

print('Local repository:', LOCAL_REPOSITORY)


## 2. Sanity check: do your external YAML files exist?

Human has a packaged default, but for mouse and pig you should have local `*_externals_modified.yml` files.


In [ ]:
expected = [
    LOCAL_REPOSITORY / 'homo_sapiens_externals_modified.yml',
    LOCAL_REPOSITORY / 'mus_musculus_externals_modified.yml',
    LOCAL_REPOSITORY / 'sus_scrofa_externals_modified.yml',
]
for p in expected:
    print(('OK' if p.exists() else 'MISSING').ljust(8), p.name)


If a file is missing:
- go back to `prepare_new_external_yaml.ipynb`
- generate the template and create the `_modified.yml` file


## 3. Build a graph snapshot

The canonical pattern is:

1. resolve organism name
2. pick snapshot release
3. `api.build_graph(...)`
4. inspect + reuse

We do this for each organism below. If you only need one organism, run only that section.


### 3A. Human (homo_sapiens)


In [ ]:
organism, latest_release = api.resolve_organism('human')
SNAPSHOT_RELEASE = latest_release  # pin to a specific release if needed
organism, SNAPSHOT_RELEASE


In [ ]:
# Build (or load) the graph snapshot
# - calculate_caches=True speeds up later queries (slower build, faster use).
api.build_graph(organism_name=organism, snapshot_release=SNAPSHOT_RELEASE, calculate_caches=True)


In [ ]:
# Quick inspection
g = api.track.graph
print('Organism:', g.graph.get('organism'))
print('Snapshot release:', g.graph.get('ensembl_release'))
print('Main assembly:', g.graph.get('genome_assembly'))
print('Nodes:', g.number_of_nodes())
print('Edges:', g.number_of_edges())

aed = sorted(getattr(g, 'available_external_databases', []))
print('External DBs enabled (count):', len(aed))
print('External DBs (first 20):', aed[:20])


In [ ]:
# Where is the graph file stored?
sorted(LOCAL_REPOSITORY.glob('graph_homo_sapiens*.pickle'))[-5:]


### 3B. Mouse (mus_musculus)


In [ ]:
organism, latest_release = api.resolve_organism('mus musculus')
SNAPSHOT_RELEASE = latest_release
organism, SNAPSHOT_RELEASE


In [ ]:
api.build_graph(organism_name=organism, snapshot_release=SNAPSHOT_RELEASE, calculate_caches=True)


In [ ]:
g = api.track.graph
print('Organism:', g.graph.get('organism'))
print('Snapshot release:', g.graph.get('ensembl_release'))
print('Main assembly:', g.graph.get('genome_assembly'))
print('Nodes:', g.number_of_nodes())
print('Edges:', g.number_of_edges())

aed = sorted(getattr(g, 'available_external_databases', []))
print('External DBs enabled (count):', len(aed))
print('External DBs (first 20):', aed[:20])


In [ ]:
sorted(LOCAL_REPOSITORY.glob('graph_mus_musculus*.pickle'))[-5:]


### 3C. Pig (sus_scrofa)


In [ ]:
organism, latest_release = api.resolve_organism('sus scrofa')
SNAPSHOT_RELEASE = latest_release
organism, SNAPSHOT_RELEASE


In [ ]:
api.build_graph(organism_name=organism, snapshot_release=SNAPSHOT_RELEASE, calculate_caches=True)


In [ ]:
g = api.track.graph
print('Organism:', g.graph.get('organism'))
print('Snapshot release:', g.graph.get('ensembl_release'))
print('Main assembly:', g.graph.get('genome_assembly'))
print('Nodes:', g.number_of_nodes())
print('Edges:', g.number_of_edges())

aed = sorted(getattr(g, 'available_external_databases', []))
print('External DBs enabled (count):', len(aed))
print('External DBs (first 20):', aed[:20])


In [ ]:
sorted(LOCAL_REPOSITORY.glob('graph_sus_scrofa*.pickle'))[-5:]


## 4. Advanced knobs (when builds are slow or you need control)

If you need more control than `api.build_graph(...)`, you can instantiate `idtrack.Track` directly and
forward arguments to the underlying graph builder (for example to force a rebuild).

Example pattern (advanced):

```python
dm = api.get_database_manager(organism_name='homo_sapiens', snapshot_release=115)
track = idtrack.Track(dm, create_even_if_exist=True, overwrite_even_if_exist=True)
api.track = track  # attach it back to the API if you want
```

Use this only when you know you want to override the cached graph.
